# Rolling Statistical Anomaly Detector

This notebook implements the second anomaly-detection approach for the CATS dataset: a rolling statistical detector.

Unlike the rule-based detector, which uses fixed thresholds derived from the nominal period, a rolling statistical detector evaluates observations relative to recent signal behaviour.

The detector will use rolling statistics calculated from historical observations to identify unusually large deviations in the telemetry signals.

The implementation will preserve temporal ordering and avoid using ground-truth anomaly labels or metadata to generate predictions.

The detector will be evaluated against the same CATS ground truth and compared with the rule-based baseline using observation-level and event-level metrics.

## Step 2 — Definition the Rolling Detection Rule

The rolling statistical detector will use a moving window of recent observations to estimate the local behaviour of each telemetry signal.

For each signal, the detector will calculate a rolling mean and rolling standard deviation using historical observations. An observation will be flagged when its value falls sufficiently far from the rolling mean relative to the rolling standard deviation.

The detector will use a minimum history requirement before generating predictions. This avoids making decisions from an insufficient number of observations.

To preserve temporal ordering, the rolling statistics will be shifted so that the observation being evaluated is not included in the statistics used to evaluate itself.

The window size and deviation threshold will be selected explicitly and documented before evaluation. Ground-truth anomaly labels and metadata will not be used to select them.

## Step 3 — Set the Initial Detector Parameters

The initial rolling statistical detector will use a 60-observation window, equivalent to 60 seconds because CATS is sampled at 1 Hz.

For each telemetry signal, the detector will calculate the rolling mean and rolling standard deviation from the preceding 60 observations. The statistics will be shifted by one observation so that the current observation is excluded from its own reference window.

An observation will be flagged when its value is more than three rolling standard deviations above or below the historical rolling mean.

A complete 60-observation history will be required before the detector generates a prediction.

These parameters establish an initial baseline. They are not assumed to be optimal, and their performance will be evaluated using the CATS ground truth after predictions have been generated.

## Step 4 — Load the Dataset and Define the Signal Inputs

This step loads the CATS dataset and defines the telemetry signals that the rolling statistical detector will evaluate.

The dataset contains 5,000,000 timestamped observations. The first 1,000,000 observations form the documented nominal period, followed by the evaluation period.

The detector will use only the 17 telemetry signals as inputs. The ground-truth anomaly label and category will be retained for later evaluation and excluded from the detector inputs.

The timestamp index will preserve the temporal order required for calculating rolling statistics.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cats" / "data.parquet"

df = pd.read_parquet(DATA_PATH)

signal_columns = [
    "aimp", "amud", "arnd", "asin1", "asin2",
    "adbr", "adfl", "bed1", "bed2", "bfo1",
    "bfo2", "bso1", "bso2", "bso3", "ced1",
    "cfo1", "cso1",
]

print("Dataset shape:", df.shape)
print("Timestamp index:", isinstance(df.index, pd.DatetimeIndex))
print("Signals defined:", len(signal_columns))
print("Signal columns present:", all(
    column in df.columns for column in signal_columns
))
print("Missing values in signal columns:", df[signal_columns].isna().sum().sum())

Dataset shape: (5000000, 19)
Timestamp index: True
Signals defined: 17
Signal columns present: True
Missing values in signal columns: 0


### Conclusion

The CATS dataset loaded successfully with 5,000,000 observations and a datetime index.

All 17 telemetry signals are present, and the original signal data contains no missing values.

The input definition explicitly excludes the ground-truth label and category from the detector features. This establishes the correct inputs for implementing the rolling statistical detector.

The dataset is ready for the next step: calculating rolling means and standard deviations using historical observations only.

## Step 5 — Calculate Historical Rolling Statistics

This step calculates a rolling mean and rolling standard deviation for each telemetry signal using the preceding 60 observations.

The rolling statistics will be shifted by one observation so that the current observation is not included in the reference values used to evaluate it.

A complete 60-observation history is required before the statistics are available. Predictions will not be generated in this step.

The resulting rolling statistics will be used in the next step to calculate deviations and generate anomaly predictions.

### Mathematical Foundation — Rolling Statistical Detection

The rolling statistical detector compares each current observation with the recent behaviour of its signal. It uses the previous 60 observations to estimate normal behaviour and flags values that deviate substantially from that estimate.

**1. Rolling mean**

The rolling mean is the average of the previous 60 observations:

$$
\mu_t = \frac{1}{60}\sum_{i=1}^{60}x_{t-i}
$$

Here, $x_{t-i}$ represents an earlier observation, and $\mu_t$ is the historical average at time $t$. The current observation, $x_t$, is excluded.

**2. Rolling standard deviation**

The rolling standard deviation measures how much the previous observations vary around their average:

$$
s_t = \sqrt{\frac{1}{59}\sum_{i=1}^{60}(x_{t-i}-\mu_t)^2}
$$

A small standard deviation indicates that recent values are close together. A large standard deviation indicates greater variation. The denominator is $59$ because Pandas calculates the sample standard deviation by default.

**3. Measure the deviation**

The detector compares the current observation with the historical average:

$$
z_t = \frac{x_t-\mu_t}{s_t}
$$

The value $z_t$ expresses the difference in units of standard deviation. For example, $z_t=2$ means the observation is two standard deviations above the historical average, while $z_t=-3$ means it is three standard deviations below the average.

**4. Decide whether an observation is anomalous**

The initial threshold is three standard deviations. An observation is flagged when:

$$
|z_t| > 3
$$

Equivalently, when $s_t>0$, the observation is flagged if it falls outside these limits:

$$
\begin{aligned}
\text{Lower limit} &= \mu_t - 3s_t\\
\text{Upper limit} &= \mu_t + 3s_t
\end{aligned}
$$

The detector applies these limits independently to all 17 telemetry signals. If at least one signal falls outside its limits, the observation is flagged as anomalous.

**Implementation considerations**

- **Historical data only:** The rolling statistics are shifted by one observation, ensuring the current value does not influence its own reference values.
- **Minimum history:** The detector requires 60 previous observations before calculating its reference statistics.
- **Zero standard deviation:** If a signal has been constant throughout the previous 60 observations, its standard deviation is zero. The implementation must handle this case without dividing by zero.
- **Initial threshold:** Three standard deviations is a starting choice, not a guarantee of a specific false-alarm rate. Actual performance must be measured against the dataset's ground-truth labels after predictions are generated.

In [2]:
WINDOW_SIZE = 60

rolling_mean = (
    df[signal_columns]
    .rolling(window=WINDOW_SIZE, min_periods=WINDOW_SIZE)
    .mean()
    .shift(1)
)

rolling_std = (
    df[signal_columns]
    .rolling(window=WINDOW_SIZE, min_periods=WINDOW_SIZE)
    .std()
    .shift(1)
)

print("Rolling mean shape:", rolling_mean.shape)
print("Rolling standard deviation shape:", rolling_std.shape)
print("First observation with rolling statistics:", rolling_mean.dropna().index.min())
print("Missing rolling means:", rolling_mean.isna().sum().sum())
print("Missing rolling standard deviations:", rolling_std.isna().sum().sum())

Rolling mean shape: (5000000, 17)
Rolling standard deviation shape: (5000000, 17)
First observation with rolling statistics: 2023-01-01 00:01:00
Missing rolling means: 1020
Missing rolling standard deviations: 1020


### Conclusion

The rolling mean and rolling standard deviation have been calculated for all 17 telemetry signals using the preceding 60 observations.

The first 60 observations do not have enough historical data to calculate these statistics. The first usable statistics occur at `2023-01-01 00:01:00`, using the 60 observations from the preceding minute.

The rolling mean represents the recent average value of a signal, while the rolling standard deviation measures how much its recent values vary.

In the next step, the detector will use these two statistics to establish an acceptable range for each signal. If a current observation falls below or above that range, it will be flagged as a potential anomaly.

No anomaly predictions have been generated yet.

## Step 6 — Generate Anomaly Predictions

The rolling statistical detector will now compare each telemetry observation with the historical limits calculated in the previous step.

For each signal, the detector establishes a lower and upper limit using its rolling mean and rolling standard deviation:

$$
\begin{aligned}
\text{Lower limit} &= \text{Rolling mean} - 3 \times \text{Rolling standard deviation} \\
\text{Upper limit} &= \text{Rolling mean} + 3 \times \text{Rolling standard deviation}
\end{aligned}
$$

An observation is flagged when at least one of the 17 signals falls outside its corresponding limits.

The implementation will use these limits directly rather than divide by the rolling standard deviation. This also avoids division by zero when a signal's recent values have no variation.

The first 60 observations do not have a complete historical window and will not be flagged by this rule. Ground-truth labels will not be used to generate predictions.

In [3]:
lower_bounds = rolling_mean -3 * rolling_std
upper_bounds = rolling_mean +3 * rolling_std

below_lower = df[signal_columns] < lower_bounds
above_upper = df[signal_columns] > upper_bounds

rolling_predictions = (below_lower | above_upper).any(axis=1)


print("Total observations:", len(rolling_predictions))
print("Predicted anomalies:", int(rolling_predictions.sum()))
print("Predicted normal observations:", int((~rolling_predictions).sum()))
print("Predicted anomaly rate:", round(rolling_predictions.mean(), 4))
print("Predictions generated:", rolling_predictions.notna().all())

Total observations: 5000000
Predicted anomalies: 1317817
Predicted normal observations: 3682183
Predicted anomaly rate: 0.2636
Predictions generated: True


### Conclusion

The rolling statistical detector generated predictions for all 5,000,000 observations. It flagged 1,317,817 observations (26.36%) as potential anomalies and classified the remaining 3,682,183 observations as normal.

An observation is flagged when at least one of the 17 telemetry signals falls outside its historical limits, calculated using the preceding 60 observations.

These results describe the detector's predictions but do not establish their accuracy. The next step will restrict the predictions to the evaluation period and compare them with the ground-truth labels to measure detection performance.

## Step 7 — Evaluate Predictions Against Ground Truth

The detector's predictions must be compared with the ground-truth anomaly labels to assess its performance.

The first 1,000,000 observations form the nominal period, while the remaining 4,000,000 observations form the evaluation period. The evaluation will therefore use only the final 4,000,000 observations.

The ground-truth label `y` identifies whether each observation is normal or anomalous. It will be used only to evaluate the predictions, not to generate them.

### 7.1 Confusion Matrix

The confusion matrix compares the detector's predictions with the actual labels and produces four outcomes:

| Outcome | Meaning |
|---|---|
| **True Positive (TP)** | An anomaly was correctly detected. |
| **False Positive (FP)** | A normal observation was incorrectly flagged as anomalous. |
| **True Negative (TN)** | A normal observation was correctly identified. |
| **False Negative (FN)** | An actual anomaly was missed. |

These four outcomes are used to calculate the evaluation metrics below.

### 7.2 Precision

Precision measures the proportion of predicted anomalies that are actually anomalous.

$$
\text{Precision} = \frac{TP}{TP+FP}
$$

A higher precision means fewer false alarms among the observations flagged as anomalous.

### 7.3 Recall

Recall measures the proportion of actual anomalies that the detector successfully identifies.

$$
\text{Recall} = \frac{TP}{TP+FN}
$$

A higher recall means the detector misses fewer actual anomalies.

### 7.4 False-Alarm Rate

The false-alarm rate measures the proportion of actual normal observations incorrectly flagged as anomalous.

$$
\text{False-Alarm Rate} = \frac{FP}{FP+TN}
$$

A lower false-alarm rate means fewer normal observations are incorrectly flagged.

### 7.5 Evaluation Plan

We will calculate precision, recall, and false-alarm rate for the rolling statistical detector. We will then compare these results with the rule-based baseline using the same 4,000,000-observation evaluation period.

The objective is to understand the trade-off between detecting actual anomalies and avoiding false alarms. No single metric is sufficient to describe the detector's overall performance.

In [7]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

evaluation_df = df.iloc[1_000_000:]

y_true = evaluation_df["y"].astype(int)
y_pred = rolling_predictions.iloc[1_000_000].astype(int)

tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
).ravel()

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

print("Evaluation observations:", len(evaluation_df))
print("Ground-truth anomalies:", int(y_true.sum()))
print("Confusion matrix:")
print([[tn, fp], [fn, tp]])

InvalidParameterError: The 'y_pred' parameter of confusion_matrix must be an array-like. Got np.int64(0) instead.